Integrate wb97m reference energies and compute metrics for the torsion dataset

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy.stats import pearsonr
import os


# Path Settings
results_path = "./Torsion_Net-results.csv"
min_data_path = "./data/wb97m/torsionnet500-wb97m.csv"

# Data Loading 
print("Loading data...")
df = pd.read_csv(results_path)
df_min = pd.read_csv(min_data_path)

# Grouping identifier: detect new molecules based on scan start value
SCAN_COL = "scan"
df["scan_group_id"] = (df[SCAN_COL] == -165).cumsum()

# Calculate relative energies within each group (subtract group minimum)
df["u_qm_rel"] = df.groupby("scan_group_id")["Reference Energy"].transform(lambda x: x - x.min())

# Selecting specific columns for matching to save memory
df_min_subset = df_min[['name_1', 'name_2', 'scan', 'wb97m']]

# Left join to ensure all rows in results.csv are preserved
df_final = pd.merge(
    df, 
    df_min_subset, 
    left_on=['mol_id', 'conf_id', SCAN_COL], 
    right_on=['name_1', 'name_2', 'scan'], 
    how='left'
)
df_final = df_final.drop(columns=['name_1', 'name_2'])


# Metrics Calculation
valid_df = df_final.dropna(subset=['ResFF Energy', 'wb97m'])

if not valid_df.empty:
    y_pred = valid_df['ResFF Energy']
    y_true = valid_df['wb97m']

    # Performance metrics
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    pearson, _ = pearsonr(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    print("\n" + "="*45)
    print("       RESFF ENERGY FITTING STATISTICS")
    print("="*45)
    print(f"MAE:                 {mae:.4f} kcal/mol")
    print(f"RMSE:                {rmse:.4f} kcal/mol")
    print(f"Pearson r:           {pearson:.4f}")
    print(f"R² Score:            {r2:.4f}")
    print("="*45 + "\n")
else:
    print("\nWARNING: No matched rows found. Please check if IDs in both CSVs are consistent.")

# Saving Results
out_path = results_path.replace(".csv", "-wb97m.csv")
df_final.to_csv(out_path, index=False)
print(f"File saved to: {out_path}")

Loading data...
Calculating relative energies (group-wise normalization)...
Performing multi-key merge based on mol_id, conf_id, and scan...

       RESFF ENERGY FITTING STATISTICS
Sample Size (N):     12000
MAE:                 0.4526 kcal/mol
RMSE:                0.7151 kcal/mol
Pearson r:           0.9654
R² Score:            0.9303

Aligned data with metrics saved to:
/home/datahouse1/jiangxinyu/ResFF-github/Torsion_Net-results_stats.csv


Integrate wb97m reference energies and compute metrics for the s66x8 dataset

In [ ]:
import pandas as pd

# Path Settings
path_ref = "./data/wb97m/s66_wb97m.csv"
path_target = "./s66x8-results.csv"
out_path = path_target.replace(".csv", "-wb97m.csv")

# Load data
df_ref = pd.read_csv(path_ref)
df_tar = pd.read_csv(path_target)

df_ref_sub = df_ref[["mol_name", "Reference Energy", "scan", "wb97m Energy"]]
df_final = df_tar.merge(
    df_ref_sub, 
    on=["mol_name", "Reference Energy"], 
    how="left"
)

# Extract molecule base name
df_final["mol_base"] = df_final["mol_name"].str.replace(r"\d{3}$", "", regex=True)

def calculate_relative_energies(group):
    group = group.copy()
    
    group["wb97m Energy"] = group["wb97m Energy"] * 627.5
    group["wb97m_rel"] = group["wb97m Energy"] - group["wb97m Energy"].min()
    group["ResFF_rel"] = group["ResFF Energy"] - group["ResFF Energy"].min()
    
    return group

df_final = df_final.groupby("mol_base", group_keys=False).apply(calculate_relative_energies)

# Metrics Calculation
valid_df = df_final.dropna(subset=['ResFF_rel', 'wb97m_rel'])

if not valid_df.empty:
    y_pred = valid_df['ResFF_rel']
    y_true = valid_df['wb97m_rel']

    # Performance metrics
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    pearson, _ = pearsonr(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    print("\n" + "="*45)
    print("       RESFF ENERGY FITTING STATISTICS")
    print("="*45)
    print(f"MAE:                 {mae:.4f} kcal/mol")
    print(f"RMSE:                {rmse:.4f} kcal/mol")
    print(f"Pearson r:           {pearson:.4f}")
    print(f"R² Score:            {r2:.4f}")
    print("="*45 + "\n")
else:
    print("\nWARNING: No matched rows found. Please check if IDs in both CSVs are consistent.")

# Saving Results
out_path = results_path.replace(".csv", "-wb97m.csv")
df_final.to_csv(out_path, index=False)
print(f"File saved to: {out_path}")